# CG — alanine dipeptide (gas, 300 K)

Coarse-graining and FRESEAN workflow for alanine dipeptide, using the same all-atom trajectory as `02_MD-alanine-dipeptide-gas-300K.ipynb` (`input_data/MD-gas-300K/`).

| Step | What happens | This notebook |
|------|--------------|---------------|
| Coarse-graining | load AA `topol`/`traj`, map to COM beads | `CoarseGrain.cg_universe()` (§1) |
| FRESEAN (CG) | correlation matrix + modes on CG trajectory | §2 |
| FRESEAN (all-atom) | same frames and spectral parameters | §3 |
| **VDoS comparison** | overlay CG vs all-atom total VDoS | §4 |
| Back-mapping | CG modes → all-atom | `backmap_modes()` (§5) |
| PLUMED prep | mode direction PDB | `write_plumed_mode_input()` (§6) |

With four CG beads FRESEAN yields twelve eigenmodes; modes **7 and 8** (1-based) are back-mapped here, as in `05_CG-hewl.ipynb`. If a mode index is missing from the CG spectrum, the all-atom FRESEAN result is used instead.


In [ ]:
import shutil
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import pandas as pd

from pyfresean import Align, CoarseGrain, FRESEAN
from pyfresean.postprocess import low_frequency_peaks, mode_spectra, plot_spectra

METAD_DATA = Path("fresean_metaD_data")
INPUT_DATA = Path("input_data")
INPUT_DATA.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA = Path("output_data")
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)
AA = INPUT_DATA / "MD-gas-300K"
OUT = OUTPUT_DATA / "MD-gas-300K-CG"
OUT.mkdir(parents=True, exist_ok=True)

for name in ("plumed-mode-projection.dat",):
    src = METAD_DATA / name
    if src.exists():
        shutil.copy2(src, OUT / name)

MODE_NUMBERS = (7, 8)
MODE_INDICES = tuple(m - 1 for m in MODE_NUMBERS)

N_FRAMES = 5000
DT = 0.004  # ps between frames
N_CORR = 100
SIGMA = 10.0
N_CONSTRAINTS_AA = 6


def download_if_needed(path, url):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {path} ...")
        urllib.request.urlretrieve(url, path)


print(f"All-atom data: {AA.resolve()}")
print(f"Output directory: {OUT.resolve()}")
print(f"Using modes {MODE_NUMBERS} (0-based indices {MODE_INDICES})")


## 1. Coarse-grain from all-atom files

Each residue maps to a **BACK** (backbone COM) bead and, except GLY/ACE/NME, a **SIDE** (sidechain COM) bead.

`CoarseGrain.cg_universe(topology, trajectory, select=...)` loads the all-atom topology and trajectory, applies the selection, and returns `(cg, u_cg)`. Pass `output_top` with a `.top` extension to save bead masses (a companion `.gro` is written automatically). Trajectory files download from the same URLs as notebook 02 if they are not already under `input_data/MD-gas-300K/`.


In [ ]:
topol = AA / "topol.tpr"
traj = AA / "traj.trr"
ref = INPUT_DATA / "harmonic-normal-modes" / "min.xyz"

download_if_needed(
    topol,
    "https://www.dropbox.com/scl/fi/zux7notw4x9omhqj1rngj/topol.tpr?rlkey=svz7x62n0g4sooee10vp968zp&dl=1",
)
download_if_needed(
    traj,
    "https://www.dropbox.com/scl/fi/vhbtq7x3vqdap5t3i0vlw/traj.trr?rlkey=cffmtm0tysiy85bd3luaa46af&dl=1",
)

u_ref = mda.Universe(str(ref))
ref_positions = u_ref.atoms.positions.copy()

cg_top = OUT / "topol-cg.top"
cg_gro = OUT / "topol-cg.gro"
cg_traj = OUT / "traj-cg.trr"
ref_aa_pdb = OUT / "ref.pdb"
ref_cg_pdb = OUT / "ref-cg.pdb"

for p in (cg_top, cg_gro, cg_traj):
    if p.exists():
        p.unlink()

cg, u_cg = CoarseGrain.cg_universe(
    topol,
    traj,
    select="all",
    stop=N_FRAMES,
    output_top=str(cg_top),
    output_traj=str(cg_traj),
    in_memory=True,
)

aa_sel = cg.atomgroup
print(f"All-atom atoms: {aa_sel.n_atoms}")
print(f"Source trajectory frames: {len(aa_sel.universe.trajectory)} (using first {N_FRAMES})")

m = cg.mapping
rows = []
for i in range(m.n_beads):
    rows.append(
        {
            "bead": i,
            "type": m.bead_types[i],
            "resname": m.resnames[i],
            "resindex": int(m.resindices[i]),
            "mass": m.bead_masses[i],
            "n_atoms": len(m.atom_indices[i]),
        }
    )

print(f"CG beads: {m.n_beads}")
print(f"n_constraints: {m.n_constraints}")
print(f"Bead masses: {u_cg.atoms.masses}")
pd.DataFrame(rows)

# Shared alignment reference: harmonic minimum mapped to CG COM beads.
ref_cg = cg.map_positions(ref_positions)

aa_sel.positions = ref_positions.astype(np.float32)
aa_sel.write(str(ref_aa_pdb))
with open(ref_aa_pdb, "r+", encoding="utf-8") as handle:
    content = handle.read()
    handle.seek(0)
    if not content.startswith("REMARK TYPE=OPTIMAL"):
        handle.write("REMARK TYPE=OPTIMAL\n" + content)

u_cg.trajectory[0]
saved_frame0 = u_cg.atoms.positions.copy()
u_cg.atoms.positions = ref_cg.astype(np.float32)
cg.write_topology(u_cg, str(ref_cg_pdb))
u_cg.atoms.positions = saved_frame0

assert len(u_cg.trajectory) == N_FRAMES
print(f"CG beads: {u_cg.atoms.n_atoms}, frames: {len(u_cg.trajectory)}")


## 2. Align CG trajectory and run FRESEAN

Alignment to CG COM positions mapped from `min.xyz` (same reference as §3). `n_corr=100`, `sigma=10` cm⁻¹.


In [ ]:
u_cg.trajectory.add_transformations(
    Align(u_cg.atoms, reference_positions=ref_cg, place_com_in_box=False),
)

analysis_cg = FRESEAN(
    u_cg,
    select="all",
    n_constraints=cg.mapping.n_constraints,
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
)
analysis_cg.run(stop=N_FRAMES)

freqs_cg = analysis_cg.results.freqs
vdos_cg = analysis_cg.results.vdos_total
eigenvalues_cg = analysis_cg.results.eigenvalues
eigenvectors_cg = analysis_cg.results.eigenvectors
corr_cg = analysis_cg.results.corr_matrix
avg_temp_cg = analysis_cg.results.avg_temperature

low_peaks_cg = low_frequency_peaks(freqs_cg, vdos_cg, max_freq=200.0)
zero_freq_idx_cg = int(low_peaks_cg[0])

print(f"T (CG) = {avg_temp_cg:.1f} K")
print(f"CG modes available: {eigenvectors_cg.shape[1]}")
print(f"low-frequency peaks (cm-1): {freqs_cg[low_peaks_cg]}")


## 3. FRESEAN on the all-atom trajectory

Same frame window, spectral parameters, and `min.xyz` alignment as §2.


In [ ]:
u_aa_fresh = mda.Universe(str(topol), str(traj))
sel_aa = u_aa_fresh.select_atoms("all")
u_aa_fresh.trajectory.add_transformations(
    Align(sel_aa, reference_positions=ref_positions, place_com_in_box=False),
)

analysis_aa = FRESEAN(
    u_aa_fresh,
    select="all",
    n_constraints=N_CONSTRAINTS_AA,
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
)
analysis_aa.run(stop=N_FRAMES)

freqs_aa = analysis_aa.results.freqs
vdos_aa = analysis_aa.results.vdos_total
avg_temp_aa = analysis_aa.results.avg_temperature
low_peaks_aa = low_frequency_peaks(freqs_aa, vdos_aa, max_freq=200.0)

print(f"T (all-atom) = {avg_temp_aa:.1f} K")
print(f"low-frequency peaks (cm-1): {freqs_aa[low_peaks_aa]}")


## 4. Compare CG vs all-atom VDoS

Overlay of total vibrational density of states (0–200 cm⁻¹) on the same axes.

Both trajectories are aligned to the same harmonic minimum (`min.xyz`; CG via mapped COM beads). CG bead velocities only retain mass-weighted COM motion (~10% of all-atom kinetic energy); high-frequency internal vibrations are folded out. FRESEAN normalizes each spectrum by its own inferred temperature and degrees of freedom (6 for CG vs 60 for all-atom), so amplitudes are not strictly comparable. CG is usually lower at physical peaks; excess at the 0 cm⁻¹ bin reflects the FFT zero-frequency component after alignment.


In [ ]:
if not np.allclose(freqs_aa, freqs_cg):
    raise ValueError("AA and CG frequency grids differ; check n_corr, dt, and sigma.")

fig, ax = plt.subplots(figsize=(7, 4))
plot_spectra(
    freqs_aa,
    [vdos_aa, vdos_cg],
    ax=ax,
    xlim=(0, 200),
    labels=["All-atom VDoS", "CG VDoS"],
    colors=["black", "tab:orange"],
    linestyles=["-", "--"],
    vlines=freqs_aa[low_peaks_aa].tolist(),
    title="CG vs all-atom total VDoS",
)
plt.tight_layout()
plt.show()


## 5. Back-map modes 7 and 8 to all-atom coordinates

Mass-weighted backmap: `u_i = u_b * sqrt(m_i / M_b)` for each atom `i` in bead `b`. Modes are taken from the CG zero-frequency bin when available; otherwise from all-atom FRESEAN at the same bin.


In [ ]:
eigenvectors_aa = analysis_aa.results.eigenvectors
zero_freq_idx_aa = int(low_peaks_aa[0])
masses = cg._atom_masses()
n_atoms = cg.atomgroup.n_atoms

aa_modes = []
for mode_num, mode_idx in zip(MODE_NUMBERS, MODE_INDICES):
    if mode_idx < eigenvectors_cg.shape[1]:
        cg_mode = eigenvectors_cg[zero_freq_idx_cg, mode_idx]
        aa_mode = cg.backmap_modes(cg_mode)
        source = "CG backmap"
        cg_norm = np.linalg.norm(cg_mode)
    elif mode_idx < eigenvectors_aa.shape[1]:
        weighted = eigenvectors_aa[zero_freq_idx_aa, mode_idx].reshape(n_atoms, 3)
        aa_mode = weighted / np.sqrt(masses)[:, np.newaxis]
        source = "all-atom FRESEAN"
        cg_norm = np.linalg.norm(weighted)
    else:
        raise ValueError(
            f"Mode {mode_num} not available: CG has {eigenvectors_cg.shape[1]} modes, "
            f"all-atom has {eigenvectors_aa.shape[1]}."
        )
    aa_modes.append(aa_mode)
    print(
        f"Mode {mode_num} ({source}): norm={cg_norm:.4f}, "
        f"max |AA|={np.max(np.linalg.norm(aa_mode, axis=1)):.4f}"
    )

mode_vdos_cg = mode_spectra(
    corr_cg,
    eigenvectors_cg[zero_freq_idx_cg, list(MODE_INDICES)],
)
fig, ax = plt.subplots(figsize=(6, 4))
for i, mode_num in enumerate(MODE_NUMBERS):
    plot_spectra(freqs_cg, mode_vdos_cg[i], ax=ax, labels=f"CG mode {mode_num}")
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()


## 6. PLUMED mode input

`write_plumed_mode_input()` writes `plumed-mode-input.pdb` (reference structure plus modes 7 and 8). For the full HEWL metadynamics workflow see `05_CG-hewl.ipynb`.


In [ ]:
plumed_input = OUT / "plumed-mode-input.pdb"
cg.write_plumed_mode_input(aa_modes, str(plumed_input))

for mode_num, aa_mode in zip(MODE_NUMBERS, aa_modes):
    cg.write_plumed_direction_pdb(aa_mode, str(OUT / f"evec_{mode_num}_aa_scaled.pdb"))

print(f"Wrote {plumed_input}")
